In [ ]:
!pip install tellurium
!pip install numpy==1.24.4

In [ ]:
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code
# Wood smoke exposure model with continuous exposure to simulate epithelial and cytokine dynamics
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
'''

# Experimental data for pro-inflammatory cytokines
time_exp = np.array([5, 24, 48, 72])
P_exp = np.array([0.0363, 0.4545, 0.5636, 1])

# Create the model
r = te.loada(continuous_exposure_model)

# Simulate the model
result = r.simulate(0, 1000, 10000)



# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create a single plot
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot simulation results
sns.lineplot(data=simulation_df, x='Time', y='Healthy Epithelial Cells', ax=ax1, label='Healthy Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Damaged Epithelial Cells', ax=ax1, label='Damaged Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Pro-inflammatory Cytokines', ax=ax1, label='Pro-inflammatory Cytokines')

# Set axis labels
ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)

# Set x-axis limits to match the time of interest
ax1.set_xlim(0, 1000)
ax1.legend(fontsize=12)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Save the figure in both .svg and .eps formats
plt.savefig('epithelial_baseline.png', dpi=300, bbox_inches='tight')
plt.savefig('epithelial_baseline.svg', format='svg', bbox_inches='tight')
plt.savefig('epithelial_baseline.eps', format='eps', dpi=300, bbox_inches='tight')

# Adjust layout and display the plot
plt.tight_layout()

plt.show()

In [ ]:
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code
# Wood smoke exposure model with continuous exposure to simulate epithelial and cytokine dynamics
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
'''

# Experimental data for pro-inflammatory cytokines
time_exp = np.array([5, 24, 48, 72])
P_exp = np.array([0.0363, 0.4545, 0.5636, 1])

# Create the model
r = te.loada(continuous_exposure_model)

# Simulate the model
result = r.simulate(0, 1000, 10000)

# Extract simulation results
time_sim = result[:, 0]
E_h_sim = result[:, 1]
E_d_sim = result[:, 2]
P_sim = result[:, 3]

# Normalize P_sim to compare with experimental data
P_sim_normalized = P_sim / np.max(P_sim)

# After simulating and normalizing the data, create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Healthy Epithelial Cells': E_h_sim,
    'Damaged Epithelial Cells': E_d_sim,
    'Pro-inflammatory Cytokines': P_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'Experimental Data': P_exp
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create a single plot
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot simulation results
sns.lineplot(data=simulation_df, x='Time', y='Healthy Epithelial Cells', ax=ax1, label='Healthy Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Damaged Epithelial Cells', ax=ax1, label='Damaged Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Pro-inflammatory Cytokines', ax=ax1, label='Pro-inflammatory Cytokines')

# Set axis labels
ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)

# Set x-axis limits to match the time of interest
ax1.set_xlim(0, 1000)
ax1.legend(fontsize=12)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Save the figure in both .svg and .eps formats
plt.savefig('epithelial_baseline.png', dpi=300, bbox_inches='tight')
plt.savefig('epithelial_baseline.svg', format='svg', bbox_inches='tight')
plt.savefig('epithelial_baseline.eps', format='eps', dpi=300, bbox_inches='tight')

# Adjust layout and display the plot
plt.tight_layout()

plt.show()

In [ ]:
r.reset()
r.steadyState() # Bring the model to steady state using a steady state solver
print(r.getSteadyStateValuesNamedArray())

In [ ]:
print(r.getSteadyStateSolver())

In [ ]:
#what if we look at the SUM of the experimental data for both TNF-alpha and IL-6 versus prediction? next, what about both plotted separately?
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
'''

# Experimental data (SUM of TNF-alpha and IL-6 data from Calcabrini et al., normalized to % of max)
time_exp = np.array([5, 24, 48, 72])
P_exp = np.array([0.045, 0.459, 0.685, 1])

# Create the model
r = te.loada(continuous_exposure_model)

# Simulate the model
result = r.simulate(0, 73, 1000)

# Extract simulation results
time_sim = result[:, 0]
E_h_sim = result[:, 1]
E_d_sim = result[:, 2]
P_sim = result[:, 3]

# Normalize P_sim to compare with experimental data
P_sim_normalized = P_sim / np.max(P_sim)

# After simulating and normalizing the data, create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Healthy Epithelial Cells': E_h_sim,
    'Damaged Epithelial Cells': E_d_sim,
    'Pro-inflammatory Cytokines': P_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'Experimental Data': P_exp
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14))

# Plot simulation results
sns.lineplot(data=simulation_df, x='Time', y='Healthy Epithelial Cells', ax=ax1, label='Healthy Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Damaged Epithelial Cells', ax=ax1, label='Damaged Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Pro-inflammatory Cytokines', ax=ax1, label='Pro-inflammatory Cytokines')
ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)
ax1.set_xlim(0, 72)
ax1.legend(fontsize=12)
ax1.tick_params(axis='both', which='major', labelsize=12)

# Plot experimental data vs. simulated P
sns.scatterplot(data=experimental_df, x='Time', y='Experimental Data', ax=ax2, s=100, label='Experimental Data')
sns.lineplot(data=simulation_df, x='Time', y='Normalized Cytokines', ax=ax2, label='Simulated P (normalized)')
ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('Normalized Concentration', fontsize=14)
ax2.legend(fontsize=12)
ax2.set_xlim(0, 73)
ax2.tick_params(axis='both', which='major', labelsize=12)

plt.savefig('simulation_vs_experimental.png', dpi=300, bbox_inches='tight')
plt.savefig('simulation_vs_experimental.eps', format='eps', bbox_inches='tight')
# Adjust layout and display the plot
plt.tight_layout()

plt.show()

In [ ]:
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code remains the same
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
'''

# Experimental data
time_exp = np.array([5, 24, 48, 72])
TNF_alpha_exp = np.array([0.07, 0.07, 1.0, 0.07])
IL6_exp = np.array([0.0363, 0.4545, 0.5636, 1.0])
P_exp_sum = np.array([0.045, 0.459, 0.685, 1])  # Sum of normalized TNF-alpha and IL-6

# Create the model and simulate
r = te.loada(continuous_exposure_model)
result = r.simulate(0, 73, 1000)

# Extract and normalize simulation results
time_sim = result[:, 0]
P_sim = result[:, 3]
P_sim_normalized = P_sim / np.max(P_sim)

# Create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'TNF-alpha': TNF_alpha_exp,
    'IL-6': IL6_exp,
    'Sum': P_exp_sum
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# First figure - Individual cytokine levels
plt.figure(figsize=(10, 7))
bar_width = 0.35
index = np.arange(len(time_exp))
plt.bar(index - bar_width/2, TNF_alpha_exp, bar_width, label='TNF-alpha', alpha=0.8)
plt.bar(index + bar_width/2, IL6_exp, bar_width, label='IL-6', alpha=0.8)

plt.tick_params(axis='both', which='major', labelsize=12)
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Normalized Concentration', fontsize=14)
plt.xticks(index, time_exp)
plt.legend(fontsize=12)
plt.ylim(0, 1.1)

# Add value labels on top of each bar
for i, v in enumerate(TNF_alpha_exp):
    plt.text(i - bar_width/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=12)
for i, v in enumerate(IL6_exp):
    plt.text(i + bar_width/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('figure6a.eps', format='eps', dpi=300, bbox_inches='tight')
plt.show()

# Second figure - Experimental Sum vs. Simulated P
plt.figure(figsize=(10, 7))
sns.scatterplot(data=experimental_df, x='Time', y='Sum', s=100, label='TNF-alpha + IL-6')
sns.lineplot(data=simulation_df, x='Time', y='Normalized Cytokines', label='Simulated P (normalized)')
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Normalized Concentration', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.legend(fontsize=12)
plt.xlim(0, 73)

plt.tight_layout()
plt.savefig('figure6b.eps', format='eps', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code
# Wood smoke exposure model with continuous exposure to simulate epithelial and cytokine dynamics for steady state analysis and validation to data
#produced Fig4 in manuscript
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;
  R9: -> A; e8*E_h
  R10: A -> ; e9*A
end
'''

# Experimental data for pro-inflammatory cytokines
time_exp = np.array([5, 24, 48, 72])
P_exp = np.array([0.0363, 0.4545, 0.5636, 1])

# Create the model
r = te.loada(continuous_exposure_model)

# Simulate the model
result = r.simulate(0, 1000, 10000)

# Extract simulation results
time_sim = result[:, 0]
E_h_sim = result[:, 1]
E_d_sim = result[:, 2]
P_sim = result[:, 3]

# Normalize P_sim to compare with experimental data
P_sim_normalized = P_sim / np.max(P_sim)

# After simulating and normalizing the data, create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Healthy Epithelial Cells': E_h_sim,
    'Damaged Epithelial Cells': E_d_sim,
    'Pro-inflammatory Cytokines': P_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'Experimental Data': P_exp
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create a single plot
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot simulation results
sns.lineplot(data=simulation_df, x='Time', y='Healthy Epithelial Cells', ax=ax1, label='Healthy Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Damaged Epithelial Cells', ax=ax1, label='Damaged Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Pro-inflammatory Cytokines', ax=ax1, label='Pro-inflammatory Cytokines')

# Set axis labels
ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)

# Set x-axis limits to match the time of interest
ax1.set_xlim(0, 1000)
ax1.legend(fontsize=12)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Save the figure in both .svg and .eps formats
plt.savefig('epithelial_baseline.png', dpi=300, bbox_inches='tight')
plt.savefig('epithelial_baseline.svg', format='svg', bbox_inches='tight')
plt.savefig('epithelial_baseline.eps', format='eps', dpi=300, bbox_inches='tight')

# Adjust layout and display the plot
plt.tight_layout()

plt.show()